# Exploratory Analysis: Mortgage Credit Stress Testing

This notebook is for telling the story of the mortgage portfolio after the Spark pipeline runs. It reads the small CSV outputs from `outputs/` and creates simple visuals for the final report or presentation.

Run `bash run.sh` first so the output folders exist.

In [ ]:
import glob
import os
import pandas as pd
import matplotlib.pyplot as plt

def read_spark_csv_folder(path):
    files = glob.glob(os.path.join(path, 'part-*.csv'))
    if not files:
        raise FileNotFoundError(f'No Spark CSV part file found in {path}')
    return pd.read_csv(files[0])


## 1. Delinquency Distribution

This chart shows how many loan records are current, 30 days delinquent, 60 days delinquent, or 90+ days delinquent. It gives a quick sense of the portfolio's credit quality.

In [ ]:
delinq = read_spark_csv_folder('../outputs/eda/delinquency_counts')
display(delinq)

plt.figure(figsize=(8, 4))
plt.bar(delinq['current_loan_delinquency'].astype(str), delinq['count'])
plt.title('Loan Records by Delinquency Status')
plt.xlabel('Current Loan Delinquency')
plt.ylabel('Record Count')
plt.show()


## 2. Risk State Over Time

This view shows how the portfolio changes by reporting month. A good story to tell here is whether serious delinquency is stable, rising, or concentrated in a few months.

In [ ]:
risk_month = read_spark_csv_folder('../outputs/eda/risk_state_by_month')
display(risk_month.head())

pivot = risk_month.pivot_table(index='reporting_month', columns='risk_state', values='count', fill_value=0)
pivot.plot(kind='bar', stacked=True, figsize=(10, 5))
plt.title('Risk State Distribution by Reporting Month')
plt.xlabel('Reporting Month')
plt.ylabel('Record Count')
plt.xticks(rotation=45)
plt.show()


## 3. Credit Score Band and Serious Delinquency

This chart connects borrower credit quality to serious delinquency. It helps explain why credit score is a useful model feature.

In [ ]:
credit = read_spark_csv_folder('../outputs/eda/credit_score_band_delinquency')
display(credit)

plt.figure(figsize=(8, 4))
plt.bar(credit['credit_score_band'], credit['serious_delinquency_rate'])
plt.title('Serious Delinquency Rate by Credit Score Band')
plt.xlabel('Credit Score Band')
plt.ylabel('Serious Delinquency Rate')
plt.xticks(rotation=30)
plt.show()


## 4. LTV Band and Serious Delinquency

LTV is important in mortgage risk because higher-LTV loans may have less borrower equity. This chart helps show whether that pattern appears in the data.

In [ ]:
ltv = read_spark_csv_folder('../outputs/eda/ltv_band_delinquency')
display(ltv)

plt.figure(figsize=(8, 4))
plt.bar(ltv['ltv_band'], ltv['serious_delinquency_rate'])
plt.title('Serious Delinquency Rate by Original LTV Band')
plt.xlabel('Original LTV Band')
plt.ylabel('Serious Delinquency Rate')
plt.show()


## 5. Transition Matrix

The transition matrix shows loan migration from one risk state to the next. This is one of the strongest credit-risk parts of the project.

In [ ]:
transition = read_spark_csv_folder('../outputs/migration/transition_matrix_pivot')
display(transition)

transition_plot = transition.set_index('from_state')
plt.figure(figsize=(8, 5))
plt.imshow(transition_plot.fillna(0))
plt.title('Transition Matrix Heatmap')
plt.xlabel('To State')
plt.ylabel('From State')
plt.xticks(range(len(transition_plot.columns)), transition_plot.columns, rotation=45)
plt.yticks(range(len(transition_plot.index)), transition_plot.index)
plt.colorbar(label='Transition Probability')
plt.show()


## 6. Baseline vs Stress Expected Loss

This chart is the main stress testing result. It compares expected loss under the baseline macro conditions and the stressed macro assumptions.

In [ ]:
scenario = read_spark_csv_folder('../outputs/scenarios/scenario_comparison')
display(scenario)

plt.figure(figsize=(7, 4))
plt.bar(scenario['scenario'], scenario['total_expected_loss'])
plt.title('Total Expected Loss: Baseline vs Stress')
plt.xlabel('Scenario')
plt.ylabel('Total Expected Loss')
plt.show()


## 7. Model Metrics

The MLlib model metrics are stored in JSON. Use these values in the final report and slides.

In [ ]:
import json

with open('../outputs/ml/model_metrics.json', 'r') as file:
    metrics = json.load(file)

metrics


In [ ]:
import glob
import os
from functools import reduce

from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, col, count, date_format, to_date, when


def create_eda_outputs(modeling_df, fred_2025):
    """
    QUESTION ANSWERED:
    What patterns in the portfolio should be understood before training the model?
    """

    # How many records are in each delinquency status?
    delinquency_counts = modeling_df.groupBy("current_loan_delinquency") \
        .count() \
        .orderBy("current_loan_delinquency")

    delinquency_counts.show()
    delinquency_counts.coalesce(1).write.mode("overwrite") \
        .option("header", "true") \
        .csv("outputs/eda/delinquency_counts")

    # How does the portfolio move between risk states over time?
    risk_state_by_month = modeling_df.groupBy("reporting_month", "risk_state") \
        .count() \
        .orderBy("reporting_month", "risk_state")

    risk_state_by_month.coalesce(1).write.mode("overwrite") \
        .option("header", "true") \
        .csv("outputs/eda/risk_state_by_month")

    # Which states contain the largest number of mortgage records?
    portfolio_by_state = modeling_df.groupBy("property_state") \
        .count() \
        .orderBy(col("count").desc())

    portfolio_by_state.show()
    portfolio_by_state.coalesce(1).write.mode("overwrite") \
        .option("header", "true") \
        .csv("outputs/eda/portfolio_by_state")

    # How does serious delinquency vary by credit score band?
    credit_score_risk = modeling_df.groupBy("credit_score_band") \
        .agg(
            count("*").alias("loan_records"),
            avg("is_seriously_delinquent").alias("serious_delinquency_rate"),
        ) \
        .orderBy("credit_score_band")

    credit_score_risk.coalesce(1).write.mode("overwrite") \
        .option("header", "true") \
        .csv("outputs/eda/credit_score_band_delinquency")

    # How does serious delinquency vary by original LTV band?
    ltv_risk = modeling_df.groupBy("ltv_band") \
        .agg(
            count("*").alias("loan_records"),
            avg("is_seriously_delinquent").alias("serious_delinquency_rate"),
        ) \
        .orderBy("ltv_band")

    ltv_risk.coalesce(1).write.mode("overwrite") \
        .option("header", "true") \
        .csv("outputs/eda/ltv_band_delinquency")

    # Save the monthly macro data so it can be graphed in the notebook.
    fred_2025.coalesce(1).write.mode("overwrite") \
        .option("header", "true") \
        .csv("outputs/eda/fred_2025")

    # Spark SQL question:
    # By month, how large is the portfolio and what is its serious delinquency rate?
    modeling_df.createOrReplaceTempView("mortgage_portfolio")

    monthly_portfolio_summary = spark.sql("""
        SELECT
            reporting_month,
            COUNT(*) AS loan_records,
            SUM(current_actual_upb) AS total_current_upb,
            AVG(borrower_credit_score) AS average_credit_score,
            AVG(original_ltv) AS average_original_ltv,
            AVG(is_seriously_delinquent) AS serious_delinquency_rate
        FROM mortgage_portfolio
        GROUP BY reporting_month
        ORDER BY reporting_month
    """)

    monthly_portfolio_summary.show(truncate=False)
    monthly_portfolio_summary.coalesce(1).write.mode("overwrite") \
        .option("header", "true") \
        .csv("outputs/sql/monthly_portfolio_summary")